# Matemáticas de la Inteligencia Artificial
## Sesión 4 — Funciones de pérdida, derivadas y descenso de gradiente

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/04_gradiente/laboratorio.ipynb)

### Pregunta de la sesión
**¿Cómo sabe una máquina en qué dirección debe modificar sus parámetros?**

En la sesión anterior comprobamos que una red puede **representar** XOR, pero no sabíamos encontrar automáticamente sus pesos. En este laboratorio construiremos, desde cero, la cadena

$$
\text{modelo}\longrightarrow\text{pérdida}\longrightarrow\text{gradiente}\longrightarrow\text{actualización}.
$$

Trabajaremos solo con **NumPy** y **Matplotlib**. No usaremos `sklearn`, PyTorch ni `autograd`. Ejecuta las celdas de arriba abajo y completa los `TODO`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. De error cualitativo a función de pérdida

Para una predicción $\widehat y$ y un objetivo $y$ usamos primero

$$\ell(\widehat y,y)=\frac12(\widehat y-y)^2.$$

El número resultante no solo dice si hemos fallado: mide **cuánto** nos hemos alejado del objetivo.

### Diccionario matemática–código

| Matemática | Python |
|---|---|
| $\theta$ | `theta` |
| $\mathcal L(\theta)$ | `loss(theta)` |
| $\nabla\mathcal L$ | `grad(theta)` |
| $\eta$ | `eta` |
| $\theta\leftarrow\theta-\eta\nabla L$ | `theta = theta - eta * grad` |

In [ ]:
def perdida_cuadratica(y_hat, y):
    # TODO: implementa 0.5*(y_hat-y)**2
    return ...

y = 1.0
for y_hat in [1.0, 0.8, 0.5, 0.0, -1.0]:
    print(f'ŷ={y_hat:4.1f}  pérdida={perdida_cuadratica(y_hat, y)}')

# Pregunta: ¿qué información conserva esta pérdida que no conserva el simple acierto/fallo?

## 2. Paisaje de pérdida, derivada y aproximación local

Para la neurona $\widehat y=wx$ con un único ejemplo $x=2$, $y=1$,

$$\mathcal L(w)=\frac12(2w-1)^2,\qquad \mathcal L'(w)=4w-2.$$

La derivada permite construir el modelo local

$$\mathcal L(w_0+h)\approx\mathcal L(w_0)+\mathcal L'(w_0)h.$$

También podemos comprobar una derivada mediante la diferencia centrada

$$f'(x)\approx\frac{f(x+h)-f(x-h)}{2h}.$$

In [ ]:
def L1(w):
    return 0.5*(2*w-1)**2

def dL1(w):
    # TODO: derivada analítica
    return ...

def derivada_centrada(f, x, h=1e-5):
    # TODO: [f(x+h)-f(x-h)]/(2*h)
    return ...

ws = np.linspace(-0.5, 1.5, 400)
plt.figure(figsize=(7,4))
plt.plot(ws, L1(ws))
plt.xlabel('w'); plt.ylabel('L(w)'); plt.grid(alpha=.25)
plt.title('Paisaje de pérdida')
plt.show()

w0, h = 0.0, 0.1
print('L(w0)=', L1(w0))
print('derivada=', dL1(w0))
print('aproximación local=', L1(w0)+dL1(w0)*h)
print('valor exacto=', L1(w0+h))
print('derivada numérica en w=0.3=', derivada_centrada(L1, 0.3))

## 3. Dos parámetros: gradiente y Hessiano

Para el modelo $\widehat y=wx+b$ con datos $(0,1)$ y $(1,3)$,

$$\mathcal L(w,b)=\frac14[(b-1)^2+(w+b-3)^2],$$

$$\nabla\mathcal L(w,b)=\begin{pmatrix}\tfrac12(w+b-3)\\\tfrac12(w+2b-4)\end{pmatrix}.$$

La curvatura está descrita por

$$H=\begin{pmatrix}1/2&1/2\\1/2&1\end{pmatrix}.$$

Sus autovalores indican la curvatura en las direcciones propias.

In [ ]:
def L2(theta):
    w, b = theta
    return 0.25*((b-1)**2 + (w+b-3)**2)

def grad_L2(theta):
    w, b = theta
    # TODO: completa las dos componentes
    return np.array([..., ...], dtype=float)

H = np.array([[0.5,0.5],[0.5,1.0]])
# TODO: usa np.linalg.eigvalsh
autovalores = ...
print('grad L(0,0)=', grad_L2(np.array([0.,0.])))
print('autovalores=', autovalores)

w_grid = np.linspace(-1,4,180)
b_grid = np.linspace(-1,3,180)
W,B = np.meshgrid(w_grid,b_grid)
Z = 0.25*((B-1)**2+(W+B-3)**2)
plt.figure(figsize=(7,6))
plt.contour(W,B,Z,levels=18)
plt.scatter([2],[1],s=70,label='mínimo')
plt.xlabel('w'); plt.ylabel('b'); plt.grid(alpha=.2); plt.legend()
plt.title('Curvas de nivel de L(w,b)')
plt.show()

## 4. Descenso de gradiente y `learning rate`

La regla es

$$\theta_{n+1}=\theta_n-\eta\nabla\mathcal L(\theta_n).$$

Para $L(w)=\tfrac12(2w-1)^2$ se obtiene exactamente

$$w_{n+1}-\frac12=(1-4\eta)\left(w_n-\frac12\right),$$

y por tanto la estabilidad exige $0<\eta<0.5$. Vamos a provocar convergencia suave, convergencia oscilatoria, oscilación permanente y divergencia.

In [ ]:
def descenso_1d(w0, eta, n_pasos=12):
    w = float(w0)
    historia = [w]
    for _ in range(n_pasos):
        # TODO: w <- w - eta*dL1(w)
        w = ...
        historia.append(w)
    return np.array(historia)

for eta in [0.1,0.3,0.5,0.6]:
    hist = descenso_1d(0.0, eta)
    plt.figure(figsize=(7,4))
    plt.plot(range(len(hist)), hist, marker='o')
    plt.axhline(0.5, linestyle='--', label='w*=0.5')
    plt.xlabel('iteración'); plt.ylabel('w'); plt.grid(alpha=.25); plt.legend()
    plt.title(f'eta={eta}')
    plt.show()

# Completa: 0.1 -> ..., 0.3 -> ..., 0.5 -> ..., 0.6 -> ...

In [ ]:
def descenso_2d(theta0, eta=0.5, n_pasos=15):
    theta = np.array(theta0, dtype=float)
    historia = [theta.copy()]
    perdidas = [L2(theta)]
    for _ in range(n_pasos):
        # TODO: actualiza theta con grad_L2
        theta = ...
        historia.append(theta.copy())
        perdidas.append(L2(theta))
    return np.array(historia), np.array(perdidas)

trayectoria, perdidas = descenso_2d([0.,0.], eta=.5)
# TODO: representa la pérdida frente a la iteración.

plt.figure(figsize=(7,6))
plt.contour(W,B,Z,levels=18)
plt.plot(trayectoria[:,0], trayectoria[:,1], marker='o')
plt.scatter([2],[1],s=70,label='mínimo')
plt.xlabel('w'); plt.ylabel('b'); plt.grid(alpha=.2); plt.legend()
plt.title('Trayectoria en el espacio de parámetros')
plt.show()

## 5. `Gradient checking`

En varias dimensiones comprobamos cada coordenada con

$$\frac{\partial L}{\partial\theta_i}\approx\frac{L(\theta+h e_i)-L(\theta-h e_i)}{2h}.$$

Esto sirve para **verificar** un gradiente, no para entrenar una red grande: exigiría dos evaluaciones completas de la pérdida por parámetro.

In [ ]:
def gradiente_numerico(f, theta, h=1e-5):
    theta = np.array(theta, dtype=float)
    g = np.zeros_like(theta)
    for i in range(len(theta)):
        e = np.zeros_like(theta); e[i] = 1.0
        # TODO: diferencia centrada en la coordenada i
        g[i] = ...
    return g

theta_prueba = np.array([0.7,-0.2])
print('analítico:', grad_L2(theta_prueba))
print('numérico :', gradiente_numerico(L2, theta_prueba))

## 6. Pérdida logística: una señal suave para clasificación

Con etiquetas $y\in\{-1,+1\}$ y *score* $z=\mathbf w^T\mathbf x+b$,

$$\ell(z,y)=\log(1+e^{-yz}),$$

$$\nabla_{\mathbf w}\ell=-\frac{y}{1+e^{yz}}\mathbf x,\qquad \frac{\partial\ell}{\partial b}=-\frac{y}{1+e^{yz}}.$$

Un ejemplo bien clasificado con margen grande genera una corrección pequeña; uno mal clasificado genera una señal mayor.

In [ ]:
m = np.linspace(-6,6,400)
plt.figure(figsize=(7,4))
plt.plot(m, np.log(1+np.exp(-m)))
plt.xlabel('margen m=yz'); plt.ylabel('pérdida'); plt.grid(alpha=.25)
plt.title('Pérdida logística')
plt.show()

x=np.array([2.,1.]); y=1.; w=np.array([-0.5,0.]); b=0.; eta=.1
z=w@x+b
# TODO: factor = y/(1+exp(y*z))
factor = ...
w_nuevo = ...
b_nuevo = ...
print('score antiguo=',z)
print('score nuevo=',w_nuevo@x+b_nuevo)

# 7. Problema final — Entrenar una neurona logística desde cero

La solución completa queda reservada al cuaderno docente. No uses `sklearn`, PyTorch, `autograd` ni un optimizador preconstruido.

### A. Datos y geometría
1. Representa los seis datos. 2. Propón visualmente una frontera posible.

### B. Modelo y pérdida
3. Implementa $z=Xw+b$. 4. Implementa la pérdida logística media.

### C. Gradiente
5. Implementa el gradiente analítico respecto de $w$ y $b$. 6. Compruébalo con diferencias finitas.

### D. Entrenamiento
7. Inicializa $w=0$, $b=0$. 8. Entrena con descenso de gradiente. 9. Compara `eta=0.01`, `0.1` y `1.0`. 10. Representa las curvas de pérdida.

### E. Resultado e interpretación
11. Dibuja la frontera final. 12. Calcula las seis predicciones. 13. Explica qué significa que el modelo haya aprendido. 14. Distingue parámetros e hiperparámetros. 15. Explica qué problema aparecerá al pasar a una red multicapa.

In [ ]:
X_final=np.array([[-2.,-1.],[-1.5,-2.],[-1.,-1.2],[1.,1.1],[1.5,2.],[2.2,1.]])
y_final=np.array([-1.,-1.,-1.,1.,1.,1.])

# TODO A: representa los datos.

def scores(X,w,b):
    return ...

def perdida_logistica_media(X,y,w,b):
    return ...

def gradiente_logistico(X,y,w,b):
    # Devuelve (grad_w, grad_b).
    return ...

def entrenar_logistico(X,y,eta,n_pasos=200):
    # Inicializa, actualiza y guarda la pérdida.
    return ...

# TODO C: gradient checking.
# TODO D: compara los tres learning rates.
# TODO E: frontera, predicciones e interpretación.

## Cierre

Debes poder explicar y programar

$$\boxed{\text{modelo}\to\mathcal L(\theta)\to\nabla\mathcal L(\theta)\to\theta_{n+1}=\theta_n-\eta\nabla\mathcal L(\theta_n)}.$$

La sesión 5 parte del obstáculo que queda abierto: en una red multicapa, ¿cómo calculamos eficientemente el gradiente de **todos** los pesos? Esa necesidad conducirá a **backpropagation**.